# 03 — Anotación y Preparación del Dataset

Carga de máscaras LabelMe, visualización y armado del dataset para nnU-Net/SMP.

In [ ]:
import os, json, cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA_PATH   = Path(os.environ.get("MICROCIRCULATION_DATA", "/content/drive/MyDrive/microcirculation"))
FRAMES_DIR  = DATA_PATH / "frames_for_annotation"
MASKS_DIR   = DATA_PATH / "masks"            # máscaras PNG generadas por LabelMe
DATASET_DIR = DATA_PATH / "dataset_nnunet"   # formato nnU-Net
MASKS_DIR.mkdir(exist_ok=True)
DATASET_DIR.mkdir(exist_ok=True)


## 3.1 Instrucciones de anotación con LabelMe

```bash
pip install labelme
labelme  # abre GUI
```

**Flujo:**
1. Abrir cada frame de `frames_for_annotation/`
2. Usar herramienta **Polygon** para trazar cada vaso visible
3. Label: `vessel` para todos los vasos
4. Exportar: File → Save → guarda JSON de LabelMe
5. Convertir a máscara PNG (siguiente celda)

**Criterios de anotación** (seguir consenso ESICM):
- Anotar **todos** los vasos visibles, incluyendo los de flujo intermitente
- No anotar artefactos (burbujas, reflejos)
- Vasos < 20 µm: trazar línea fina; 20–50 µm: polígono completo


## 3.2 Convertir JSON LabelMe → máscaras PNG binarias

In [ ]:
def labelme_json_to_mask(json_path: str, out_path: str) -> None:
    """Convierte anotación LabelMe a máscara binaria PNG."""
    with open(json_path) as f:
        data = json.load(f)

    h, w  = data['imageHeight'], data['imageWidth']
    mask  = np.zeros((h, w), dtype=np.uint8)

    for shape in data['shapes']:
        if shape['label'] == 'vessel':
            pts = np.array(shape['points'], dtype=np.int32)
            cv2.fillPoly(mask, [pts], 255)

    cv2.imwrite(out_path, mask)


# Convertir todos los JSONs disponibles
json_files = list(FRAMES_DIR.rglob("*.json"))
print(f"JSONs encontrados: {len(json_files)}")

for jf in json_files:
    out = MASKS_DIR / (jf.stem + "_mask.png")
    labelme_json_to_mask(str(jf), str(out))

print(f"Máscaras generadas: {len(list(MASKS_DIR.glob('*.png')))}")


## 3.3 Visualización de pares imagen / máscara

In [ ]:
mask_files = sorted(MASKS_DIR.glob("*_mask.png"))[:6]

fig, axes = plt.subplots(len(mask_files), 3, figsize=(12, 4 * len(mask_files)))
for i, mf in enumerate(mask_files):
    img_name = mf.stem.replace("_mask", "") + ".png"
    img_path = next(FRAMES_DIR.rglob(img_name), None)

    mask = cv2.imread(str(mf), cv2.IMREAD_GRAYSCALE)

    if img_path and img_path.exists():
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        axes[i, 0].imshow(img);       axes[i, 0].set_title("Frame original")
        axes[i, 1].imshow(mask, cmap='gray'); axes[i, 1].set_title("Máscara (vasos=255)")
        overlay = img.copy()
        overlay[mask > 0] = [255, 50, 50]
        axes[i, 2].imshow(overlay);   axes[i, 2].set_title("Overlay")
    for ax in axes[i]: ax.axis('off')

plt.tight_layout(); plt.savefig(DATA_PATH / "fig_03_annotations.png", dpi=150)
plt.show()


## 3.4 Concordancia interanotador (si hay 2 anotadores)

In [ ]:
def dice_coefficient(mask1: np.ndarray, mask2: np.ndarray) -> float:
    m1 = (mask1 > 0).astype(np.uint8)
    m2 = (mask2 > 0).astype(np.uint8)
    inter = (m1 & m2).sum()
    return 2 * inter / (m1.sum() + m2.sum() + 1e-8)

# TODO: Comparar máscaras de anotador 1 vs anotador 2
# dice_scores = [dice_coefficient(mask_a1, mask_a2) for ...]
# print(f"Dice interanotador: {np.mean(dice_scores):.3f} ± {np.std(dice_scores):.3f}")
print("Pendiente: cargar máscaras del segundo anotador")


## 3.5 Armar estructura de carpetas para nnU-Net y SMP

In [ ]:
from shutil import copyfile
import re

(DATASET_DIR / "imagesTr").mkdir(exist_ok=True)
(DATASET_DIR / "labelsTr").mkdir(exist_ok=True)
(DATASET_DIR / "imagesTs").mkdir(exist_ok=True)

mask_files = sorted(MASKS_DIR.glob("*_mask.png"))
np.random.seed(42)
indices    = np.random.permutation(len(mask_files))
split_idx  = int(0.8 * len(mask_files))
train_idx  = indices[:split_idx]
test_idx   = indices[split_idx:]

for i, mf in enumerate(mask_files):
    img_name = mf.stem.replace("_mask", "") + ".png"
    img_path = next(FRAMES_DIR.rglob(img_name), None)
    if img_path is None or not img_path.exists():
        continue

    case_id = f"case_{i:04d}"
    if i in train_idx:
        copyfile(str(img_path), str(DATASET_DIR / "imagesTr" / f"{case_id}.png"))
        copyfile(str(mf),       str(DATASET_DIR / "labelsTr" / f"{case_id}.png"))
    else:
        copyfile(str(img_path), str(DATASET_DIR / "imagesTs" / f"{case_id}.png"))

n_tr = len(list((DATASET_DIR / "imagesTr").glob("*.png")))
n_ts = len(list((DATASET_DIR / "imagesTs").glob("*.png")))
print(f"Train: {n_tr}  |  Test: {n_ts}")
print(f"Dataset listo en: {DATASET_DIR}")
